# CLIP attribute ablation: descriptions and selective affect


In [1]:
%pip install transformers==4.57.1 tqdm==4.67.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 19.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 20.8 MB/s  0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.3
    Uninstalling tokenizers-0.23.3:
      Successfully uninstalled tokenizers-0.23.3
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers]
ERROR: pip's dependen

## 1. Paths and fixed settings
Edit mounts here before running. Use a new `OUTPUT_ROOT` for a changed experiment. The threshold grid is fixed before any training; do not revise it in response to test scores. Keep regional features identical to the original CLIP/COCO-HFM comparison.

In [2]:
import ast, csv, hashlib, json, os, pickle, platform, random, time
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from PIL import Image, __version__ as pillow_version
import transformers
from transformers import CLIPModel, CLIPProcessor

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
FEATURE_DIR = DATA_ROOT / 'features/resnetv2_50_a1h_in1k_finetuned'
IMAGE_DIR = DATA_ROOT / 'raw/dataset_image'
OUTPUT_ROOT = Path('/kaggle/working/clip_attribute_ablation')
CACHE_DIR = OUTPUT_ROOT / 'clip_cache'
RUN_ROOT = OUTPUT_ROOT / 'runs'
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'
CLIP_REVISION = '3d74acf9a28c67741b2f4f2ea7635f0aaf6f0268'
CLIP_PROMPT = 'a photo of {word}.'
SENTIMENT_WORDS = ('good', 'bad', 'happy', 'sad', 'angry', 'calm', 'beautiful', 'ugly')
SENTIMENT_PROMPT = 'a photo of a scene that feels {word}.'
NEUTRAL_PROMPT = 'a photo of a scene that feels neutral.'
THRESHOLDS = (0.0, 0.01, 0.02, 0.04)
SCREEN_SEED = 42
SEEDS, BATCH_SIZE, EPOCHS, PATIENCE = (42, 43, 44), 32, 8, 5
LEARNING_RATE, L2, SEQ_LEN, NUM_WORKERS = 0.001, 1e-7, 75, 4
CLIP_BATCH_SIZE = 64
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
assert SCREEN_SEED in SEEDS and len(SEEDS) >= 2
assert tuple(sorted(set(THRESHOLDS))) == THRESHOLDS and THRESHOLDS[0] == 0
assert all(np.isfinite(t) and t >= 0 for t in THRESHOLDS)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')
    os.replace(temporary, path)

def write_csv(path, fields, rows):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    with temporary.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    os.replace(temporary, path)

def freeze_json(path, value):
    path = Path(path)
    if path.exists():
        if json.loads(path.read_text()) != value:
            raise ValueError(f'Existing experiment differs: {path}. Use a new OUTPUT_ROOT.')
    else:
        write_json(path, value)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, '| PyTorch:', torch.__version__)


I0000 00:00:1794996158.160831      22 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Device: cuda | PyTorch: 2.11.0+cu130


## 2. Data, vocabulary, and preflight
Uses the original cue filtering, 75-token limit, 200-dimensional GloVe matrices, and six-row sentiment expansion. Check split IDs, labels, empty texts, and feature shapes before expensive extraction/training. Test examples participate only in label-free CLIP preparation until final evaluation.

In [3]:
class VocabularyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global references are not allowed in vocabularies')

def load_vocabulary(path):
    with path.open('rb') as stream:
        return VocabularyLoader(stream, encoding='latin1').load()

word_vocab = load_vocabulary(ASSET_ROOT / 'words/vocab')
attribute_vocab = load_vocabulary(ASSET_ROOT / 'ExtractWords/vocab')
word_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'words/vector', dtype=np.float32))
attribute_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'ExtractWords/vector', dtype=np.float32))
assert word_vocab['<pad>'] == 0
assert attribute_weights.shape == (len(attribute_vocab), 200)
assert word_weights.shape == (len(word_vocab), 200)
assert torch.isfinite(word_weights).all() and torch.isfinite(attribute_weights).all()
assert set(attribute_vocab.values()) == set(range(len(attribute_vocab)))
added_sentiment_words = [word for word in SENTIMENT_WORDS if word not in attribute_vocab]
for word in added_sentiment_words:
    vector = word_weights[word_vocab[word]].clone()
    assert vector.norm() > 0
    attribute_vocab[word] = len(attribute_vocab)
    attribute_weights = torch.cat([attribute_weights, vector[None]])

blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}
STOPWORDS = set('a an the of with in on and is it are was were be been being to from by for at as into over under after before between through during without within about against among around across up down out off then than that this these those i me my we our you your he she they them his her their its who whom which what where when why how all any both each few more most other some such no nor not only own same so too very can will just should could would do does did have has had here there again further once'.split())
candidates = sorted(word for word in attribute_vocab
                    if word.isascii() and word.isalpha() and word.islower()
                    and len(word) > 1 and word not in STOPWORDS and word not in SENTIMENT_WORDS)
assert len(candidates) >= 5
split_rows = {}
for split in ('train', 'valid2', 'test2'):
    rows = [ast.literal_eval(line) for line in (ASSET_ROOT / f'text/{split}.txt').read_text().splitlines() if line.strip()]
    rows = [row for row in rows if not blocked.intersection(row[1].split())]
    assert rows and len({str(row[0]) for row in rows}) == len(rows), f'Duplicate/empty split: {split}'
    assert all(row[-1] in (0, 1) and row[1].split() for row in rows)
    split_rows[split] = rows
split_ids = {split: {str(row[0]) for row in rows} for split, rows in split_rows.items()}
assert not (split_ids['train'] & split_ids['valid2'] or split_ids['train'] & split_ids['test2'] or split_ids['valid2'] & split_ids['test2'])
tweet_ids = sorted(set.union(*split_ids.values()))
missing = [tid for tid in tweet_ids if not (IMAGE_DIR / f'{tid}.jpg').is_file() or not (FEATURE_DIR / f'{tid}.npy').is_file()]
if missing:
    raise FileNotFoundError(f'{len(missing)} missing image/feature pairs; first ID: {missing[0]}')
for split, rows in split_rows.items():
    array = np.load(FEATURE_DIR / f'{rows[0][0]}.npy', allow_pickle=False)
    assert array.shape == (196, 2048) and array.dtype == np.float32 and np.isfinite(array).all()

asset_hashes = {name: file_hash(ASSET_ROOT / name) for name in
                ('words/vocab', 'words/vector', 'ExtractWords/vocab', 'ExtractWords/vector',
                 'text/train.txt', 'text/valid2.txt', 'text/test2.txt')}
protocol = {
    'version': 1, 'thresholds': list(THRESHOLDS), 'screen_seed': SCREEN_SEED, 'seeds': list(SEEDS),
    'selection': 'highest seed-42 validation positive-class F1; smaller threshold breaks exact ties',
    'asset_hashes': asset_hashes, 'feature_dir': str(FEATURE_DIR),
    'feature_manifests': {p.name: file_hash(p) for p in sorted((FEATURE_DIR / 'manifests').glob('*.json'))},
    'batch_size': BATCH_SIZE, 'epochs': EPOCHS, 'patience': PATIENCE, 'learning_rate': LEARNING_RATE,
    'l2': L2, 'sequence_length': SEQ_LEN, 'lr_decay': 0.8, 'dropout': 0.2, 'gradient_clipping': 5.0,
    'text_padding': 'packed_lstm_masked_mean_masked_attention',
    'expanded_attribute_vocab': attribute_vocab,
    'initial_attribute_embedding_sha256': hashlib.sha256(attribute_weights.numpy().tobytes()).hexdigest(),
    'environment': {'python': platform.python_version(), 'torch': torch.__version__,
                    'numpy': np.__version__, 'transformers': transformers.__version__,
                    'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
                    'gpu_names': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]},
}
freeze_json(OUTPUT_ROOT / 'protocol.json', protocol)
print('Splits:', {split: len(rows) for split, rows in split_rows.items()})
print('Descriptive candidates:', len(candidates), '| Added words:', added_sentiment_words)


Splits: {'train': 19816, 'valid2': 2410, 'test2': 2409}
Descriptive candidates: 996 | Added words: ['good', 'bad', 'sad', 'angry', 'calm', 'ugly']


## 3. One frozen CLIP pass, reusable across all variants
Cache the **original top five descriptions** plus **all nine affect/neutral cosine scores**. The previous selected-word cache cannot recover dropped fifth descriptions. A completed cache is hash-checked and reused; partial or incompatible caches are rejected. No sarcasm labels or tweet text enter CLIP.

In [4]:
cache_spec = {
    'model_id': CLIP_MODEL_ID, 'revision': CLIP_REVISION, 'frozen': True,
    'descriptive_prompt': CLIP_PROMPT, 'sentiment_prompt': SENTIMENT_PROMPT,
    'neutral_prompt': NEUTRAL_PROMPT, 'candidates': candidates,
    'sentiment_order': ['neutral', *SENTIMENT_WORDS], 'image_dir': str(IMAGE_DIR),
    'image_ids': tweet_ids, 'batch_size': CLIP_BATCH_SIZE,
    'transformers': transformers.__version__, 'pillow': pillow_version,
    'ranking': 'stable descending descriptive cosine; original candidate order breaks ties',
}
cache_path = CACHE_DIR / 'scores.npz'
manifest_path = CACHE_DIR / 'manifest.json'
if manifest_path.exists():
    cache_manifest = json.loads(manifest_path.read_text())
    if cache_manifest['spec'] != cache_spec or file_hash(cache_path) != cache_manifest['scores_sha256']:
        raise ValueError('Incompatible or modified CLIP cache; use a fresh OUTPUT_ROOT')
else:
    if CACHE_DIR.exists() and any(CACHE_DIR.iterdir()):
        raise FileExistsError('Incomplete CLIP cache; preserve/move it before retrying')
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
    clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID, revision=CLIP_REVISION).eval().to(DEVICE)
    clip_model.requires_grad_(False)
    clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID, revision=CLIP_REVISION, use_fast=False)
    prompts = [CLIP_PROMPT.format(word=word) for word in candidates]
    prompts += [NEUTRAL_PROMPT, *[SENTIMENT_PROMPT.format(word=word) for word in SENTIMENT_WORDS]]
    top_indices, top_scores, affect_scores = [], [], []
    with torch.inference_mode():
        vectors = []
        for start in range(0, len(prompts), CLIP_BATCH_SIZE):
            inputs = clip_processor(text=prompts[start:start + CLIP_BATCH_SIZE], padding=True, truncation=True, return_tensors='pt')
            vector = clip_model.get_text_features(**{key: value.to(DEVICE) for key, value in inputs.items()})
            vectors.append(vector / vector.norm(dim=-1, keepdim=True))
        text_vectors = torch.cat(vectors)
        assert torch.isfinite(text_vectors).all()
        for start in tqdm(range(0, len(tweet_ids), CLIP_BATCH_SIZE), desc='CLIP score cache'):
            images = []
            for tid in tweet_ids[start:start + CLIP_BATCH_SIZE]:
                with Image.open(IMAGE_DIR / f'{tid}.jpg') as image:
                    images.append(image.convert('RGB'))
            pixels = clip_processor(images=images, return_tensors='pt')['pixel_values'].to(DEVICE)
            image_vectors = clip_model.get_image_features(pixel_values=pixels)
            image_vectors = image_vectors / image_vectors.norm(dim=-1, keepdim=True)
            scores = (image_vectors @ text_vectors.T).cpu()
            if not torch.isfinite(scores).all():
                raise FloatingPointError('Non-finite CLIP similarities')
            indices = torch.argsort(scores[:, :len(candidates)], dim=1, descending=True, stable=True)[:, :5]
            top_indices.append(indices.numpy())
            top_scores.append(scores.gather(1, indices).numpy())
            affect_scores.append(scores[:, len(candidates):].numpy())
    with cache_path.with_suffix('.tmp').open('wb') as stream:
        np.savez_compressed(stream, tweet_ids=np.asarray(tweet_ids),
                            descriptive_indices=np.concatenate(top_indices),
                            descriptive_scores=np.concatenate(top_scores),
                            affect_scores=np.concatenate(affect_scores))
    os.replace(cache_path.with_suffix('.tmp'), cache_path)
    cache_manifest = {'spec': cache_spec, 'scores_sha256': file_hash(cache_path),
                      'seconds_including_model_load': time.perf_counter() - started,
                      'peak_allocated_gpu_bytes': torch.cuda.max_memory_allocated(DEVICE) if DEVICE.type == 'cuda' else None,
                      'image_preprocessing': clip_processor.image_processor.to_dict()}
    write_json(manifest_path, cache_manifest)
    del clip_model, clip_processor, vectors, vector, text_vectors, image_vectors, pixels, inputs, scores
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

with np.load(cache_path, allow_pickle=False) as cache:
    assert cache['tweet_ids'].tolist() == tweet_ids
    descriptive_indices = cache['descriptive_indices'].copy()
    descriptive_scores = cache['descriptive_scores'].copy()
    affect_scores = cache['affect_scores'].copy()
assert descriptive_indices.shape == descriptive_scores.shape == (len(tweet_ids), 5)
assert affect_scores.shape == (len(tweet_ids), 9)
assert np.isfinite(descriptive_scores).all() and np.isfinite(affect_scores).all()
assert ((descriptive_indices >= 0) & (descriptive_indices < len(candidates))).all()
assert all(len(set(row)) == 5 for row in descriptive_indices)
descriptive_words = [[candidates[index] for index in row] for row in descriptive_indices]
print('Cached images:', len(tweet_ids), '| Extraction minutes:', cache_manifest['seconds_including_model_load'] / 60)


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

CLIP score cache:   0%|          | 0/385 [00:00<?, ?it/s]

Cached images: 24635 | Extraction minutes: 35.3854133778


## 4. Attribute rules, edge-case checks, and insertion diagnostics
`None` means descriptive-only. For numeric thresholds, compare the best affective cosine score to neutral with a **strict** margin. Equal affect scores use the original candidate order. Training/validation diagnostics are shown now; test diagnostics are saved but only displayed after selection.

In [5]:
def choose_attributes(descriptions, scores, threshold):
    if len(descriptions) != 5 or len(set(descriptions)) != 5:
        raise ValueError('Expected five distinct descriptions')
    if any(word in SENTIMENT_WORDS for word in descriptions):
        raise ValueError('Descriptive and affective pools must be disjoint')
    if len(scores) != 9 or not all(np.isfinite(score) for score in scores):
        raise ValueError('Expected finite neutral plus eight affect scores')
    if threshold is not None and (not np.isfinite(threshold) or threshold < 0):
        raise ValueError('Threshold must be finite and nonnegative, or None')
    winner = max(range(1, 9), key=lambda index: scores[index])
    margin = float(scores[winner]) - float(scores[0])
    insert = threshold is not None and margin > threshold
    word = SENTIMENT_WORDS[winner - 1] if insert else None
    return ([*descriptions[:4], word] if insert else list(descriptions)), word, margin

def variant_name(threshold):
    return 'descriptive_only' if threshold is None else f'margin_{threshold:g}'.replace('.', 'p')

def select_threshold(records):
    if not records:
        raise ValueError('No validation results')
    if any(not np.isfinite(row['validation_f1']) for row in records):
        raise ValueError('Non-finite validation F1')
    return min(records, key=lambda row: (-row['validation_f1'], row['threshold']))['threshold']

def check_selection_rules():
    words = ['road', 'tree', 'car', 'sky', 'house']
    assert choose_attributes(words, [0.5] * 9, 0)[0] == words
    scores = [0.25, 0.5, 0.5, 0, 0, 0, 0, 0, 0]
    assert choose_attributes(words, scores, None)[0] == words
    assert choose_attributes(words, scores, 0)[0] == words[:4] + ['good']
    assert choose_attributes(words, scores, 0.25)[0] == words
    assert choose_attributes(words, scores, 0.125)[1] == 'good'
    assert choose_attributes(words, [0.75, *scores[1:]], 0)[0] == words
    for invalid in ([float('nan')] * 9, [0] * 8):
        try:
            choose_attributes(words, invalid, 0)
        except ValueError:
            pass
        else:
            raise AssertionError('Invalid scores accepted')
    assert select_threshold([{'threshold': 0.02, 'validation_f1': 0.8},
                             {'threshold': 0.0, 'validation_f1': 0.8}]) == 0.0
    # The zero-margin rule must exactly reproduce the original neutral-first argmax.
    rng = np.random.default_rng(7)
    for scores in rng.normal(size=(100, 9)):
        winner = int(np.argmax(scores))
        expected = words if winner == 0 else words[:4] + [SENTIMENT_WORDS[winner - 1]]
        assert choose_attributes(words, scores, 0)[0] == expected
check_selection_rules()

variant_attributes, variant_diagnostics, variant_hashes = {}, {}, {}
for threshold in (None, *THRESHOLDS):
    name = variant_name(threshold)
    attributes, selected = {}, {}
    for tid, words, scores in zip(tweet_ids, descriptive_words, affect_scores):
        chosen, affect, margin = choose_attributes(words, scores, threshold)
        assert len(chosen) == len(set(chosen)) == 5 and all(w in attribute_vocab for w in chosen)
        attributes[tid] = chosen
        selected[tid] = (affect, margin)
    directory = OUTPUT_ROOT / 'attributes' / name
    directory.mkdir(parents=True, exist_ok=True)
    content = ''.join(repr([tid, *attributes[tid]]) + '\n' for tid in tweet_ids)
    path = directory / 'extract_all'
    if path.exists() and path.read_text() != content:
        raise ValueError(f'Attribute output changed: {path}')
    if not path.exists():
        path.write_text(content)
    diagnostics = {}
    for split, ids in split_ids.items():
        counts = Counter(selected[tid][0] for tid in ids if selected[tid][0] is not None)
        diagnostics[split] = {'examples': len(ids), 'inserted': sum(counts.values()),
                              'insertion_rate': sum(counts.values()) / len(ids),
                              'affective_word_counts': dict(sorted(counts.items()))}
    write_json(directory / 'diagnostics.json', diagnostics)
    variant_attributes[name], variant_diagnostics[name], variant_hashes[name] = attributes, diagnostics, file_hash(path)
    print(name, '| train/valid insertion:', [round(diagnostics[s]['insertion_rate'], 4) for s in ('train', 'valid2')])
print('Selection-rule checks passed')


descriptive_only | train/valid insertion: [0.0, 0.0]
margin_0 | train/valid insertion: [0.8646, 0.8664]
margin_0p01 | train/valid insertion: [0.5671, 0.5755]
margin_0p02 | train/valid insertion: [0.2323, 0.2432]
margin_0p04 | train/valid insertion: [0.0069, 0.0071]
Selection-rule checks passed


## 5. Dataset and unchanged hierarchical fusion
Attribute-conditioned Bi-LSTM initialization, nine guided attentions, learned modality weighting, and the original classifier. Each variant receives fresh dataset objects and seed-matched initialization. No workers retain attributes from a previous variant.

In [6]:
class SarcasmDataset(Dataset):
    def __init__(self, split, attributes):
        self.rows = split_rows[split]
        self.tokens = torch.zeros((len(self.rows), SEQ_LEN), dtype=torch.long)
        self.attributes = torch.empty((len(self.rows), 5), dtype=torch.long)
        self.labels = torch.tensor([row[-1] for row in self.rows], dtype=torch.float32)
        for i, row in enumerate(self.rows):
            ids = [word_vocab.get(token, word_vocab['<unk>']) for token in row[1].split()[:SEQ_LEN]]
            self.tokens[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
            self.attributes[i] = torch.tensor([attribute_vocab[word] for word in attributes[str(row[0])]])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        tid = str(self.rows[index][0])
        array = np.load(FEATURE_DIR / f'{tid}.npy', allow_pickle=False)
        if array.shape != (196, 2048) or array.dtype != np.float32 or not np.isfinite(array).all():
            raise ValueError(f'Invalid regional feature array: {tid}')
        return self.tokens[index], torch.from_numpy(array), self.attributes[index], self.labels[index], tid

def make_loader(data, seed, training=False):
    return DataLoader(data, batch_size=BATCH_SIZE, shuffle=training, drop_last=training,
                      num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda',
                      generator=torch.Generator().manual_seed(seed), persistent_workers=NUM_WORKERS > 0)

class GuidedAttention(nn.Module):
    def __init__(self, raw_size, guidance_size):
        super().__init__()
        hidden_size = (raw_size + guidance_size) // 2
        self.raw = nn.Linear(raw_size, hidden_size)
        self.guidance = nn.Linear(guidance_size, hidden_size, bias=False)
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, raw, guidance, mask=None):
        hidden = torch.tanh(self.raw(raw) + self.guidance(guidance).unsqueeze(1))
        scores = self.score(hidden)
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(-1), float('-inf'))
        return scores.softmax(dim=1)

class HierarchicalFusion(nn.Module):
    def __init__(self, word_weights, attribute_weights):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(word_weights.clone(), freeze=False, padding_idx=0)
        self.attribute_embedding = nn.Embedding.from_pretrained(attribute_weights.clone(), freeze=False)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(200, 256, batch_first=True, bidirectional=True)
        self.initial_state = nn.Linear(200, 4 * 256)
        self.image_projection = nn.Linear(2048, 1024)
        self.attribute_attention = nn.Sequential(nn.Linear(200, 100), nn.ReLU(), nn.Linear(100, 1))
        sizes = [1024, 512, 200]
        self.reconstruction = nn.ModuleList([nn.ModuleList([GuidedAttention(m, n) for n in sizes]) for m in sizes])
        self.modality_scores = nn.ModuleList([nn.Sequential(nn.Linear(size, 512), nn.Tanh(), nn.Linear(512, 1)) for size in sizes])
        self.modality_projection = nn.ModuleList([nn.Sequential(nn.Linear(size, 512), nn.Tanh()) for size in sizes])
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.08, a=-0.16, b=0.16)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0.01)
        nn.init.xavier_uniform_(self.image_projection.weight)
        for name, parameter in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(parameter)
            else:
                nn.init.zeros_(parameter)
                if 'bias_ih' in name:
                    with torch.no_grad():
                        parameter[256:512].fill_(1.0)

    def forward(self, tokens, images, attributes):
        lengths = tokens.ne(0).sum(dim=1)
        if (lengths == 0).any():
            raise ValueError('Encountered a text sequence containing only padding')
        text_mask = torch.arange(tokens.size(1), device=tokens.device)[None, :] < lengths[:, None]
        attribute_raw = self.attribute_embedding(attributes)
        attribute_weights = self.attribute_attention(attribute_raw).softmax(dim=1)
        attribute_guidance = (attribute_weights * attribute_raw).sum(dim=1)
        h_forward, h_backward, c_forward, c_backward = self.initial_state(attribute_guidance).relu().chunk(4, dim=1)
        embedded = self.dropout(self.word_embedding(tokens))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed, (torch.stack([h_forward, h_backward]), torch.stack([c_forward, c_backward])))
        text_raw, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=tokens.size(1))
        text_guidance = (text_raw * text_mask.unsqueeze(-1)).sum(dim=1) / lengths.unsqueeze(1)
        image_raw = self.image_projection(images).relu()
        raw = [image_raw, text_raw, attribute_raw]
        guidance = [image_raw.mean(dim=1), text_guidance, attribute_guidance]
        reconstructed = []
        for sequence, attentions, mask in zip(raw, self.reconstruction, [None, text_mask, None]):
            weights = torch.stack([attention(sequence, guide, mask) for attention, guide in zip(attentions, guidance)]).mean(dim=0)
            reconstructed.append((weights * sequence).sum(dim=1))
        scores = torch.cat([score(vector) for score, vector in zip(self.modality_scores, reconstructed)], dim=1).softmax(dim=1)
        projected = torch.stack([projection(vector) for projection, vector in zip(self.modality_projection, reconstructed)], dim=1)
        return self.classifier((scores.unsqueeze(-1) * projected).sum(dim=1)).squeeze(-1)

    def regularization(self):
        return 0.5 * sum(parameter.square().sum() for name, parameter in self.named_parameters()
                         if parameter.requires_grad and name != 'word_embedding.weight')

@torch.no_grad()
def check_padding(model):
    was_training = model.training
    model.eval()
    device = next(model.parameters()).device
    tokens = torch.tensor([[1, 2, 0], [3, 0, 0]], device=device)
    images = torch.zeros(2, 196, 2048, device=device)
    attrs = torch.zeros(2, 5, dtype=torch.long, device=device)
    torch.testing.assert_close(model(tokens, images, attrs), model(torch.cat([tokens, tokens.new_zeros(2, 4)], dim=1), images, attrs))
    mask = tokens.ne(0)
    for attention, size in zip(model.reconstruction[1], [1024, 512, 200]):
        weights = attention(torch.zeros(2, 3, 512, device=device), torch.zeros(2, size, device=device), mask).squeeze(-1)
        assert (weights[~mask] == 0).all()
        torch.testing.assert_close(weights.sum(dim=1), torch.ones(2, device=device))
    try:
        model(torch.zeros_like(tokens), images, attrs)
    except ValueError:
        pass
    else:
        raise AssertionError('Empty texts must be rejected')
    model.train(was_training)


## 6. Validation-only trainer and metrics
Each fit saves the best validation-F1 checkpoint and its validation predictions. Screening never builds a test loader. Completed fits are reused only when their config and saved output hashes match. Positive-class weighting, L2, epoch decay, clipping, dropout, and early stopping match the original experiment.

In [7]:
METRICS = ('precision', 'recall', 'f1', 'accuracy')

def binary_metrics(labels, probabilities):
    labels, probabilities = np.asarray(labels), np.asarray(probabilities)
    if labels.ndim != 1 or labels.shape != probabilities.shape or not len(labels):
        raise ValueError('Labels/probabilities must be matching nonempty vectors')
    if not np.isin(labels, [0, 1]).all() or not np.isfinite(probabilities).all() or not ((probabilities >= 0) & (probabilities <= 1)).all():
        raise ValueError('Invalid binary labels or probabilities')
    predicted = probabilities > 0.5
    tp = int(((labels == 1) & predicted).sum())
    fp = int(((labels == 0) & predicted).sum())
    tn = int(((labels == 0) & ~predicted).sum())
    fn = int(((labels == 1) & ~predicted).sum())
    return {'precision': tp / (tp + fp) if tp + fp else 0.0,
            'recall': tp / (tp + fn) if tp + fn else 0.0,
            'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
            'accuracy': (tp + tn) / len(labels), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    labels, probabilities, ids = [], [], []
    for tokens, images, attributes, targets, batch_ids in tqdm(loader, desc='Evaluate', leave=False):
        logits = model(tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True), attributes.to(DEVICE, non_blocking=True))
        probabilities.extend(logits.sigmoid().cpu().tolist())
        labels.extend(targets.tolist())
        ids.extend(batch_ids)
    assert len(ids) == len(set(ids)) == len(loader.dataset)
    return binary_metrics(labels, probabilities), ids, labels, probabilities

def save_predictions(path, ids, labels, probabilities):
    write_csv(path, ['tweet_id', 'label', 'probability', 'prediction'],
              ({'tweet_id': tid, 'label': int(label), 'probability': float(p), 'prediction': int(p > 0.5)}
               for tid, label, p in zip(ids, labels, probabilities)))

def train_variant(name, seed):
    run_dir = RUN_ROOT / name / f'seed_{seed}'
    config = {'protocol': protocol, 'variant': name, 'seed': seed,
              'attribute_sha256': variant_hashes[name], 'clip_cache_sha256': cache_manifest['scores_sha256']}
    completion_path = run_dir / 'validation_result.json'
    if completion_path.exists():
        if json.loads((run_dir / 'config.json').read_text()) != config:
            raise ValueError(f'Run config changed: {run_dir}')
        result = json.loads(completion_path.read_text())
        for filename, digest in result['output_hashes'].items():
            if file_hash(run_dir / filename) != digest:
                raise ValueError(f'Modified completed run: {run_dir / filename}')
        print('Reusing completed fit:', name, seed)
        return result
    if run_dir.exists():
        raise FileExistsError(f'Incomplete run preserved: {run_dir}. Move it aside or choose a fresh OUTPUT_ROOT.')
    run_dir.mkdir(parents=True)
    write_json(run_dir / 'config.json', config)
    seed_everything(seed)
    train_data = SarcasmDataset('train', variant_attributes[name])
    valid_data = SarcasmDataset('valid2', variant_attributes[name])
    positive = train_data.labels.sum().item()
    if not 0 < positive < len(train_data) or len(train_data) < BATCH_SIZE:
        raise ValueError('Training requires both classes and at least one full batch')
    positive_weight = (len(train_data) - positive) / positive
    train_loader = make_loader(train_data, seed, training=True)
    valid_loader = make_loader(valid_data, seed)
    base_model = HierarchicalFusion(word_weights, attribute_weights).to(DEVICE)
    check_padding(base_model)
    network = nn.DataParallel(base_model) if torch.cuda.device_count() > 1 else base_model
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
    optimizer = torch.optim.Adam(base_model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
    best_f1, stale_epochs, history = -1.0, 0, []
    started = time.perf_counter()
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(DEVICE)
    for epoch in range(1, EPOCHS + 1):
        network.train()
        epoch_loss, seen = 0.0, 0
        epoch_start = time.perf_counter()
        for tokens, images, attributes, labels, _ in tqdm(train_loader, desc=f'{name} seed {seed} epoch {epoch}'):
            tokens, images = tokens.to(DEVICE, non_blocking=True), images.to(DEVICE, non_blocking=True)
            attributes, labels = attributes.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(network(tokens, images, attributes), labels) + L2 * base_model.regularization()
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite training loss')
            loss.backward()
            nn.utils.clip_grad_norm_(base_model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += loss.item() * len(labels)
            seen += len(labels)
        valid_metrics, ids, labels, probabilities = evaluate(network, valid_loader)
        record = {'epoch': epoch, 'train_loss': epoch_loss / seen, 'validation': valid_metrics,
                  'seconds': time.perf_counter() - epoch_start}
        history.append(record)
        write_json(run_dir / 'history.json', history)
        print(name, seed, json.dumps(record))
        if valid_metrics['f1'] > best_f1:
            best_f1, stale_epochs = valid_metrics['f1'], 0
            torch.save({'model': base_model.state_dict(), 'epoch': epoch,
                        'validation_f1': best_f1, 'seed': seed}, run_dir / 'best.pt.tmp')
            os.replace(run_dir / 'best.pt.tmp', run_dir / 'best.pt')
            save_predictions(run_dir / 'validation_predictions.csv', ids, labels, probabilities)
            best_metrics, best_epoch = valid_metrics, epoch
        else:
            stale_epochs += 1
        scheduler.step()
        if stale_epochs >= PATIENCE:
            break
    result = {'variant': name, 'seed': seed, 'selected_epoch': best_epoch,
              'validation': best_metrics, 'training_seconds': time.perf_counter() - started,
              'positive_weight': positive_weight,
              'trainable_parameters': sum(p.numel() for p in base_model.parameters() if p.requires_grad),
              'peak_allocated_gpu_bytes': torch.cuda.max_memory_allocated(DEVICE) if DEVICE.type == 'cuda' else None,
              'output_hashes': {filename: file_hash(run_dir / filename) for filename in
                                ('best.pt', 'history.json', 'validation_predictions.csv')}}
    write_json(completion_path, result)
    del network, base_model, optimizer, scheduler, criterion, train_loader, valid_loader
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

assert binary_metrics([1, 0, 1, 0], [0.9, 0.8, 0.5, 0.1]) == {
    'precision': 0.5, 'recall': 0.5, 'f1': 0.5, 'accuracy': 0.5, 'tp': 1, 'fp': 1, 'tn': 1, 'fn': 1}
assert binary_metrics([0, 1], [0.1, 0.5])['f1'] == 0


## 7. Screen thresholds on validation and freeze selection
This is the only threshold-selection step. Zero is included so stricter replacement is not assumed to help. The best seed-42 validation checkpoint for each candidate is compared. Screening is exploratory and may overfit validation; three final seeds describe training variability, not independent replication of threshold selection.

In [8]:
screening = []
for threshold in THRESHOLDS:
    name = variant_name(threshold)
    result = train_variant(name, SCREEN_SEED)
    screening.append({'threshold': threshold, 'variant': name, 'seed': SCREEN_SEED,
                      'validation_f1': result['validation']['f1'], 'selected_epoch': result['selected_epoch'],
                      'validation_insertion_rate': variant_diagnostics[name]['valid2']['insertion_rate']})
selected_threshold = select_threshold(screening)
selection = {'selected_threshold': selected_threshold, 'selected_variant': variant_name(selected_threshold),
             'criterion': protocol['selection'], 'screening': screening}
freeze_json(OUTPUT_ROOT / 'selection.json', selection)
write_csv(OUTPUT_ROOT / 'threshold_screening.csv', list(screening[0]), screening)
print(json.dumps(selection, indent=2))


margin_0 seed 42 epoch 1:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

margin_0 42 {"epoch": 1, "train_loss": 0.43056431802678763, "validation": {"precision": 0.7896341463414634, "recall": 0.8102189781021898, "f1": 0.7997941327843541, "accuracy": 0.8385892116182573, "tp": 777, "fp": 207, "tn": 1244, "fn": 182}, "seconds": 417.1500744969999}


margin_0 seed 42 epoch 2:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

margin_0 42 {"epoch": 2, "train_loss": 0.25288159928024295, "validation": {"precision": 0.775491113189897, "recall": 0.8644421272158499, "f1": 0.8175542406311637, "accuracy": 0.8464730290456431, "tp": 829, "fp": 240, "tn": 1211, "fn": 130}, "seconds": 171.53809656700014}


... (80 outputs left out)

margin_0p04 42 {"epoch": 6, "train_loss": 0.017304050103461848, "validation": {"precision": 0.7560521415270018, "recall": 0.8467153284671532, "f1": 0.7988194786030497, "accuracy": 0.8302904564315353, "tp": 812, "fp": 262, "tn": 1189, "fn": 147}, "seconds": 135.0319381269992}


margin_0p04 seed 42 epoch 7:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

margin_0p04 42 {"epoch": 7, "train_loss": 0.007924537462526887, "validation": {"precision": 0.8068303094983992, "recall": 0.7883211678832117, "f1": 0.7974683544303798, "accuracy": 0.8406639004149378, "tp": 756, "fp": 181, "tn": 1270, "fn": 203}, "seconds": 128.4241605569996}
{
  "selected_threshold": 0.0,
  "selected_variant": "margin_0",
  "criterion": "highest seed-42 validation positive-class F1; smaller threshold breaks exact ties",
  "screening": [
    {
      "threshold": 0.0,
      "variant": "margin_0",
      "seed": 42,
      "validation_f1": 0.8175542406311637,
      "selected_epoch": 2,
      "validation_insertion_rate": 0.866390041493776
    },
    {
      "threshold": 0.01,
      "variant": "margin_0p01",
      "seed": 42,
      "validation_f1": 0.8100641341884558,
      "selected_epoch": 4,
      "validation_insertion_rate": 0.5755186721991701
    },
    {
      "threshold": 0.02,
      "variant": "margin_0p02",
      "seed": 42,
      "validation_f1": 0.8144382308083375,

## 8. Finish three-seed training for the finalists
The threshold is now frozen. Descriptive-only and the current rule are always evaluated. A selected threshold of zero shares the current-rule results rather than duplicating fits. The screened seed-42 fits are reused.

In [9]:
selection = json.loads((OUTPUT_ROOT / 'selection.json').read_text())
roles = {'descriptive_only': variant_name(None), 'current_rule': variant_name(0.0),
         'selected_threshold': selection['selected_variant']}
finalists = list(dict.fromkeys(roles.values()))
validation_results = {}
for name in finalists:
    validation_results[name] = [train_variant(name, seed) for seed in SEEDS]
assert len({row['trainable_parameters'] for rows in validation_results.values() for row in rows}) == 1
write_json(OUTPUT_ROOT / 'finalists.json', {'roles': roles, 'seeds': list(SEEDS)})
print('All finalist fits complete. Threshold frozen at:', selection['selected_threshold'])


descriptive_only seed 42 epoch 1:   0%|          | 0/619 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
  File "/usr/lib/

descriptive_only 42 {"epoch": 1, "train_loss": 0.4877133957187078, "validation": {"precision": 0.8200692041522492, "recall": 0.7413972888425443, "f1": 0.778751369112815, "accuracy": 0.8323651452282158, "tp": 711, "fp": 156, "tn": 1295, "fn": 248}, "seconds": 130.22735541700058}


descriptive_only seed 42 epoch 2:   0%|          | 0/619 [00:00<?, ?it/s]

... (113 outputs left out)

margin_0 44 {"epoch": 7, "train_loss": 0.025268703286110247, "validation": {"precision": 0.78515625, "recall": 0.8383733055265902, "f1": 0.81089258698941, "accuracy": 0.8443983402489627, "tp": 804, "fp": 220, "tn": 1231, "fn": 155}, "seconds": 167.74165842500042}


margin_0 seed 44 epoch 8:   0%|          | 0/619 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

margin_0 44 {"epoch": 8, "train_loss": 0.017016215834793708, "validation": {"precision": 0.806282722513089, "recall": 0.8029197080291971, "f1": 0.8045977011494253, "accuracy": 0.8448132780082988, "tp": 770, "fp": 185, "tn": 1266, "fn": 189}, "seconds": 172.13250912299918}
All finalist fits complete. Threshold frozen at: 0.0


## 9. Final test evaluation
Evaluate only the frozen finalists. Prediction files use the same `tweet_id,label,probability,prediction` schema as the original notebooks. Saved metrics are recomputed from CSV predictions before aggregation. Do not go back and choose a threshold using these results.

In [10]:
def read_predictions(path, split):
    with Path(path).open(newline='') as stream:
        rows = list(csv.DictReader(stream))
    expected = {str(row[0]): int(row[-1]) for row in split_rows[split]}
    assert len(rows) == len(expected) and {row['tweet_id'] for row in rows} == set(expected)
    for row in rows:
        assert int(row['label']) == expected[row['tweet_id']]
        assert int(row['prediction']) == int(float(row['probability']) > 0.5)
    metrics = binary_metrics([int(row['label']) for row in rows], [float(row['probability']) for row in rows])
    return {row['tweet_id']: row for row in rows}, metrics

def test_variant(name, seed):
    if name not in finalists or json.loads((OUTPUT_ROOT / 'selection.json').read_text()) != selection:
        raise ValueError('Test evaluation requires frozen selection and a finalist')
    run_dir = RUN_ROOT / name / f'seed_{seed}'
    validation = json.loads((run_dir / 'validation_result.json').read_text())
    assert file_hash(run_dir / 'best.pt') == validation['output_hashes']['best.pt']
    _, checked_validation = read_predictions(run_dir / 'validation_predictions.csv', 'valid2')
    assert checked_validation == validation['validation']
    result_path = run_dir / 'results.json'
    prediction_path = run_dir / 'test_predictions.csv'
    if result_path.exists():
        result = json.loads(result_path.read_text())
        assert result['checkpoint_sha256'] == validation['output_hashes']['best.pt']
        assert result['predictions_sha256'] == file_hash(prediction_path)
    else:
        seed_everything(seed)
        base_model = HierarchicalFusion(word_weights, attribute_weights).to(DEVICE)
        checkpoint = torch.load(run_dir / 'best.pt', map_location=DEVICE, weights_only=True)
        assert checkpoint['seed'] == seed and checkpoint['epoch'] == validation['selected_epoch']
        base_model.load_state_dict(checkpoint['model'])
        network = nn.DataParallel(base_model) if torch.cuda.device_count() > 1 else base_model
        test_loader = make_loader(SarcasmDataset('test2', variant_attributes[name]), seed)
        started = time.perf_counter()
        metrics, ids, labels, probabilities = evaluate(network, test_loader)
        seconds = time.perf_counter() - started
        save_predictions(prediction_path, ids, labels, probabilities)
        result = {'variant': name, 'seed': seed, 'selected_epoch': validation['selected_epoch'],
                  'best_validation_f1': validation['validation']['f1'], 'test': metrics,
                  'training_seconds': validation['training_seconds'], 'test_seconds': seconds,
                  'trainable_parameters': validation['trainable_parameters'],
                  'checkpoint_sha256': validation['output_hashes']['best.pt'],
                  'predictions_sha256': file_hash(prediction_path)}
        write_json(result_path, result)
        del network, base_model, checkpoint, test_loader
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    _, checked_metrics = read_predictions(prediction_path, 'test2')
    assert checked_metrics == result['test']
    return result

all_results = {name: [test_variant(name, seed) for seed in SEEDS] for name in finalists}
print('Final test predictions verified for', len(finalists) * len(SEEDS), 'runs')


Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in:     self._shutdown_workers()<function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    if w.is_alive():    
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    Exception ignored in:     assert self._parent_pid == os.getpid(), 'can only test a child process'if w.is_alive():
<function _MultiProcessingDataLoaderIter._

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    Exception ignored in: self._shutdown_workers()
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
<function _MultiProcessingDataLoaderIter.__del__ at 0x7d4869e47b00>    
if w.is_alive():Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
  File "/usr/lib/python3.13/multiprocessing/process.py", line 160, in is_alive
        self._shutdown_workers()assert self._parent_pid == os.getpid(), 'can only test a child process'

AssertionError  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    : if w.is_alive():can only test a child process

  File "/usr/lib/

Final test predictions verified for 6 runs


## 10. Paper-ready summaries, paired errors, and conclusions
Metrics are stored as fractions in JSON and percentages in the summary CSV/Markdown. Differences are **percentage points**. Sample SD uses `ddof=1`; three seeds do not establish significance. Fresh `margin_0` runs provide the matched baseline; historical COCO-HFM scores belong in a separate contextual comparison, not a new matched ablation.

In [11]:
summary, table_rows, per_seed_rows = {}, [], []
for name, results in all_results.items():
    summary[name] = {metric: {'mean': float(np.mean([r['test'][metric] for r in results])),
                              'std': float(np.std([r['test'][metric] for r in results], ddof=1))}
                     for metric in METRICS}
    write_json(RUN_ROOT / name / 'summary.json', {**summary[name], 'seeds': results})
    row = {'variant': name, 'test_affective_insertion_percent': 100 * variant_diagnostics[name]['test2']['insertion_rate']}
    for metric in METRICS:
        row[f'{metric}_mean_percent'] = 100 * summary[name][metric]['mean']
        row[f'{metric}_sd_percent'] = 100 * summary[name][metric]['std']
    table_rows.append(row)
    for result in results:
        per_seed_rows.append({'variant': name, 'seed': result['seed'],
                              **{f'{metric}_percent': 100 * result['test'][metric] for metric in METRICS},
                              'selected_epoch': result['selected_epoch'],
                              'training_seconds': result['training_seconds'], 'test_seconds': result['test_seconds']})
write_json(OUTPUT_ROOT / 'summary.json', {'selection': selection, 'roles': roles, 'results': summary})
write_csv(OUTPUT_ROOT / 'summary.csv', list(table_rows[0]), table_rows)
write_csv(OUTPUT_ROOT / 'per_seed.csv', list(per_seed_rows[0]), per_seed_rows)

comparisons = [('current_minus_descriptive', roles['current_rule'], roles['descriptive_only']),
               ('selected_minus_current', roles['selected_threshold'], roles['current_rule']),
               ('selected_minus_descriptive', roles['selected_threshold'], roles['descriptive_only'])]
paired_rows, case_rows = [], []
text_by_id = {str(row[0]): row[1] for row in split_rows['test2']}
for comparison, candidate, reference in comparisons:
    for seed in SEEDS:
        candidate_rows, candidate_metrics = read_predictions(RUN_ROOT / candidate / f'seed_{seed}/test_predictions.csv', 'test2')
        reference_rows, reference_metrics = read_predictions(RUN_ROOT / reference / f'seed_{seed}/test_predictions.csv', 'test2')
        corrected, introduced = [], []
        for tid in sorted(candidate_rows):
            label = int(candidate_rows[tid]['label'])
            new_correct = int(candidate_rows[tid]['prediction']) == label
            old_correct = int(reference_rows[tid]['prediction']) == label
            if new_correct and not old_correct:
                corrected.append(tid)
            elif old_correct and not new_correct:
                introduced.append(tid)
        paired_rows.append({'comparison': comparison, 'candidate': candidate, 'reference': reference, 'seed': seed,
                            **{f'{metric}_delta_pp': 100 * (candidate_metrics[metric] - reference_metrics[metric]) for metric in METRICS},
                            'corrected': len(corrected), 'introduced': len(introduced)})
        for outcome, ids in (('corrected', corrected), ('introduced', introduced)):
            for tid in ids:
                case_rows.append({'comparison': comparison, 'seed': seed, 'outcome': outcome, 'tweet_id': tid,
                                  'text': text_by_id[tid], 'label': candidate_rows[tid]['label'],
                                  'reference_probability': reference_rows[tid]['probability'],
                                  'candidate_probability': candidate_rows[tid]['probability'],
                                  'reference_attributes': ' '.join(variant_attributes[reference][tid]),
                                  'candidate_attributes': ' '.join(variant_attributes[candidate][tid])})
write_csv(OUTPUT_ROOT / 'paired_differences.csv', list(paired_rows[0]), paired_rows)
write_csv(OUTPUT_ROOT / 'paired_cases.csv', ['comparison', 'seed', 'outcome', 'tweet_id', 'text', 'label',
          'reference_probability', 'candidate_probability', 'reference_attributes', 'candidate_attributes'], case_rows)

lines = ['# CLIP attribute ablation findings', '',
         f"Validation-selected threshold: **{selection['selected_threshold']:g}** (seed {SCREEN_SEED}).", '',
         '| Variant | Precision | Recall | F1 | Accuracy | Affect insertion (test) |',
         '| --- | ---: | ---: | ---: | ---: | ---: |']
for name in finalists:
    values = [f"{100 * summary[name][metric]['mean']:.2f} ± {100 * summary[name][metric]['std']:.2f}" for metric in METRICS]
    insertion = 100 * variant_diagnostics[name]['test2']['insertion_rate']
    lines.append('| ' + ' | '.join([name, *values, f'{insertion:.2f}%']) + ' |')
lines += ['', 'Metrics: percent, mean ± sample SD across seeds 42–44.', '', '## Paired findings']
for comparison, candidate, reference in comparisons:
    differences = [row['f1_delta_pp'] for row in paired_rows if row['comparison'] == comparison]
    accuracy = [row['accuracy_delta_pp'] for row in paired_rows if row['comparison'] == comparison]
    signs = ', '.join(f'{value:+.2f}' for value in differences)
    lines.append(f'- {candidate} minus {reference}: mean F1 {np.mean(differences):+.2f} pp; seed differences [{signs}]; mean accuracy {np.mean(accuracy):+.2f} pp.')
if roles['selected_threshold'] == roles['current_rule']:
    lines += ['', 'Validation selected the original zero-margin rule; stricter affect insertion was not preferred by this screening protocol. Selected/current results are identical, not independent runs.']
lines += ['', '## Interpretation for the conclusion',
          '- Current versus descriptive-only isolates the effect of replacing the fifth description with the selected affective word under this fixed CLIP candidate pool.',
          '- Selected versus current measures the validation-selected margin policy. A small mean gain or mixed seed signs does not establish reliable superiority.',
          '- Report precision, recall, accuracy, and insertion rates alongside F1; fewer inserted affective words alone is not evidence of better sentiment grounding.',
          '- This experiment does not independently validate affect labels, isolate CLIP pretraining from COCO, or establish out-of-distribution performance.',
          '- Use paired_cases.csv to select and visually inspect examples; prediction changes alone do not establish why the model changed its decision.',
          '- Threshold selection used one validation seed and final results share feature preparation and benchmark splits. No significance claim follows from three seeds.']
findings = '\n'.join(lines) + '\n'
(OUTPUT_ROOT / 'findings.md').write_text(findings)
print(findings)
print('All outputs:', OUTPUT_ROOT)


# CLIP attribute ablation findings

Validation-selected threshold: **0** (seed 42).

| Variant | Precision | Recall | F1 | Accuracy | Affect insertion (test) |
| --- | ---: | ---: | ---: | ---: | ---: |
| descriptive_only | 74.31 ± 1.06 | 86.06 ± 1.17 | 79.74 ± 0.10 | 82.59 ± 0.35 | 0.00% |
| margin_0 | 76.21 ± 1.94 | 84.98 ± 2.08 | 80.32 ± 0.38 | 83.42 ± 0.60 | 86.97% |

Metrics: percent, mean ± sample SD across seeds 42–44.

## Paired findings
- margin_0 minus descriptive_only: mean F1 +0.58 pp; seed differences [+0.02, +0.91, +0.82]; mean accuracy +0.83 pp.
- margin_0 minus margin_0: mean F1 +0.00 pp; seed differences [+0.00, +0.00, +0.00]; mean accuracy +0.00 pp.
- margin_0 minus descriptive_only: mean F1 +0.58 pp; seed differences [+0.02, +0.91, +0.82]; mean accuracy +0.83 pp.

Validation selected the original zero-margin rule; stricter affect insertion was not preferred by this screening protocol. Selected/current results are identical, not independent runs.

## Interpretation fo